## Convert precessed 2D slices into a 3D Image.nii and segmentation.nii for visualisation.

In [38]:
import os
import numpy as np
import nibabel as nib
from PIL import Image
import matplotlib.pyplot as plt
# # processed dataset
# img_dir = '/media/NAS06/gavinyue/genai-wsss/data/AIPFR/processed'
# save_dir = '/media/NAS06/gavinyue/genai-wsss/data/AIPFR/3d_visual'

# segmentation results
# wsss_unet
# img_dir = '../experiments/wsss_unet/results/AIPFR/pred_mask'
# save_dir = '../experiments/wsss_unet/results/AIPFR/3d_visual'

# wsss_coin
img_dir = '../experiments/wsss_coin/results/AIPFR/pred_mask'
save_dir = '../experiments/wsss_coin/results/AIPFR/3d_visual'

# full_supervised_unet
img_dir = '../experiments/full_supervised_unet/results/AIPFR/pred_mask'
save_dir = '../experiments/full_supervised_unet/results/AIPFR/3d_visual'
os.makedirs(save_dir, exist_ok=True)

In [39]:
# Get list of image files in the directory
case_dir = sorted(os.listdir(img_dir))
print(case_dir[:10])


['02_00019', '02_00020', '02_00032', '02_00037', '02_00044', '02_00055', '02_00056', '02_00058', '02_00074', '02_00079']


In [40]:
# get slice image and create 3d volume
counter = 0
for case in case_dir:
    counter += 1
    slices = sorted(os.listdir(os.path.join(img_dir, case)))
    if counter == 1:
        print(slices[:5])
    
    # Initialize a 3D numpy array
    volume = np.zeros((len(slices), 256, 256), dtype=np.uint8)
    
    for i, slice in enumerate(slices):
        slice_img = Image.open(os.path.join(img_dir, case, slice)).convert('L')
        volume[i,:,:] = np.array(slice_img)
        # plt.imshow(volume[i,:,:], cmap='gray')
        # break
    
    # Transpose the first two dimensions
    # volume = np.transpose(volume, (0, 2, 1))
    print('volume shape',volume.shape)
    # plt.imshow(volume[:,:,100], cmap='gray')
    # break
    
    # Create a NIfTI image
    nifti_img = nib.Nifti1Image(volume, np.eye(4))
    
    # Save the NIfTI image
    if 'pred' in case:
        save_name = f'{case}.nii.gz'
    else:
        save_name = f'{case}_pred.nii.gz'
    save_path = os.path.join(save_dir, save_name)
    nib.save(nifti_img, save_path)   
    
    if counter == 3: 
        break

['case02_00019_slice100_mask.png', 'case02_00019_slice101_mask.png', 'case02_00019_slice102_mask.png', 'case02_00019_slice103_mask.png', 'case02_00019_slice104_mask.png']
volume shape (378, 256, 256)
volume shape (251, 256, 256)
volume shape (422, 256, 256)


### Convert Processed 2D slices into a 3D Image.nii and segmentation.nii for visualisation.

In [ ]:
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd

def load_slices(img_dir, caseid, img_dir2 = None):
    """
    Load 2D image slices from a folder and stack them into a 3D NumPy array.
    Returns:
        np.ndarray: 3D volume array (Z, H, W).
    """
    caseid = caseid.zfill(3)  # Ensure case ID is zero-padded to 3 digits
    file_names = sorted(os.listdir(img_dir))
    
    if img_dir2:
        file_names2 = os.listdir(img_dir2)
        file_names = sorted(file_names + file_names2)
    
    file_names = [name for name in file_names if name.startswith(caseid) and name.endswith(('.png', '.jpg', '.jpeg', '.tiff'))]
    slices = []
    
    for fname in file_names:
        try:
            img = Image.open(os.path.join(img_dir, fname)).convert('L')  # Convert to grayscale
        except FileNotFoundError:
            img = Image.open(os.path.join(img_dir2, fname)).convert('L')

        slices.append(np.array(img))

    volume = np.stack(slices, axis=0)  # shape: (Z, H, W)
    return volume


def convert_to_nifti(volume_array, save_path):
    """
    Convert a 3D NumPy array (Z, H, W) to NIfTI format and save it.
    """
    # Convert NumPy array to SimpleITK Image
    volume_sitk = sitk.GetImageFromArray(volume_array)  # assumes Z, Y, X

    # Optional: set spacing (e.g., from original scan metadata)
    volume_sitk.SetSpacing([1.0, 1.0, 1.0])  # [X, Y, Z] spacing in mm

    # Save as .nii.gz
    sitk.WriteImage(volume_sitk, save_path)

# Folder with 2D slices
img_dir = '../data/OSIC/processed/fibrosis'
img_dir2 = '../data/OSIC/processed/no_fibrosis'
save_dir = '../data/OSIC/processed/3d_visual'
os.makedirs(save_dir, exist_ok=True)

# caseid = '035'
df = pd.read_csv('../data/OSIC/doctor_category.csv')
caseids_test = df['case_id'][df['test'] == 1].tolist()
caseids_test = [str(caseid).zfill(3) for caseid in caseids_test]
print(caseids_test)


for caseid in caseids_test:
    volume_array = load_slices(img_dir, caseid, img_dir2)
    save_path = os.path.join(save_dir, caseid + '_recon.nii.gz')
    convert_to_nifti(volume_array, save_path)
    print(f"Saved {caseid} to {save_path}")

['033', '064', '065', '066', '067', '127', '128', '129', '192', '193']
Saved 033 to ../data/OSIC/processed/3d_visual/033_recon.nii.gz
Saved 064 to ../data/OSIC/processed/3d_visual/064_recon.nii.gz
Saved 065 to ../data/OSIC/processed/3d_visual/065_recon.nii.gz
Saved 066 to ../data/OSIC/processed/3d_visual/066_recon.nii.gz
Saved 067 to ../data/OSIC/processed/3d_visual/067_recon.nii.gz
Saved 127 to ../data/OSIC/processed/3d_visual/127_recon.nii.gz
Saved 128 to ../data/OSIC/processed/3d_visual/128_recon.nii.gz
Saved 129 to ../data/OSIC/processed/3d_visual/129_recon.nii.gz
Saved 192 to ../data/OSIC/processed/3d_visual/192_recon.nii.gz
Saved 193 to ../data/OSIC/processed/3d_visual/193_recon.nii.gz


In [ ]:
# Save firbosis nii files, which only contained annotated fibrosis slices
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd
import sys
sys.path.append('../')
from utils.save_nii import save_as_nii

# Folder with 2D slices
img_dir = '../data/OSIC/processed/fibrosis'
save_dir = '../data/OSIC/processed/3d_visual'
os.makedirs(save_dir, exist_ok=True)

# caseid = '035'
df = pd.read_csv('../data/OSIC/doctor_category.csv')
caseids_test = df['case_id'][df['test'] == 1].tolist()
caseids_test = [str(caseid).zfill(3) for caseid in caseids_test]
print(caseids_test)


for caseid in caseids_test:
    save_path = os.path.join(save_dir, caseid + '_fibrosis.nii.gz')
    save_as_nii(caseid, save_path, img_dir)
    

['033', '064', '065', '066', '067', '127', '128', '129', '192', '193']
Saved 033 to ../data/OSIC/processed/3d_visual/033_fibrosis.nii.gz
Saved 064 to ../data/OSIC/processed/3d_visual/064_fibrosis.nii.gz
Saved 065 to ../data/OSIC/processed/3d_visual/065_fibrosis.nii.gz
Saved 066 to ../data/OSIC/processed/3d_visual/066_fibrosis.nii.gz
Saved 067 to ../data/OSIC/processed/3d_visual/067_fibrosis.nii.gz
Saved 127 to ../data/OSIC/processed/3d_visual/127_fibrosis.nii.gz
Saved 128 to ../data/OSIC/processed/3d_visual/128_fibrosis.nii.gz
Saved 129 to ../data/OSIC/processed/3d_visual/129_fibrosis.nii.gz
Saved 192 to ../data/OSIC/processed/3d_visual/192_fibrosis.nii.gz
Saved 193 to ../data/OSIC/processed/3d_visual/193_fibrosis.nii.gz


In [ ]:
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd
import sys
sys.path.append('../')
from utils.save_nii import load_slices, convert_to_nifti

# Folder with 2D slices
gt_dir = '../data/OSIC/processed/fibrosis_gt'
save_dir = '../data/OSIC/processed/3d_visual'
os.makedirs(save_dir, exist_ok=True)

# caseid = '035'
df = pd.read_csv('../data/OSIC/doctor_category.csv')
caseids_test = df['case_id'][df['test'] == 1].tolist()
caseids_test = [str(caseid).zfill(3) for caseid in caseids_test]
print(caseids_test)


for caseid in caseids_test:
    volume_array = load_slices(gt_dir, caseid)
    save_path = os.path.join(save_dir, caseid + '_gt.nii.gz')
    convert_to_nifti(volume_array, save_path)
    print(f"Saved {caseid} to {save_path}")

['033', '064', '065', '066', '067', '127', '128', '129', '192', '193']
Saved 033 to ../data/OSIC/processed/3d_visual/033_gt.nii.gz
Saved 064 to ../data/OSIC/processed/3d_visual/064_gt.nii.gz
Saved 065 to ../data/OSIC/processed/3d_visual/065_gt.nii.gz
Saved 066 to ../data/OSIC/processed/3d_visual/066_gt.nii.gz
Saved 067 to ../data/OSIC/processed/3d_visual/067_gt.nii.gz
Saved 127 to ../data/OSIC/processed/3d_visual/127_gt.nii.gz
Saved 128 to ../data/OSIC/processed/3d_visual/128_gt.nii.gz
Saved 129 to ../data/OSIC/processed/3d_visual/129_gt.nii.gz
Saved 192 to ../data/OSIC/processed/3d_visual/192_gt.nii.gz
Saved 193 to ../data/OSIC/processed/3d_visual/193_gt.nii.gz


In [ ]:
# fibrosis_pred 3d visualization for wsss_unet
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd
import sys
sys.path.append('../')
from utils.save_nii import load_slices, convert_to_nifti

# Folder with 2D slices
pred_dir = '../experiments/wsss_unet/results/OSIC/pred_mask/fibrosis_pred'
save_dir = '../data/OSIC/processed/3d_visual'
os.makedirs(save_dir, exist_ok=True)

# caseid = '035'
df = pd.read_csv('../data/OSIC/doctor_category.csv')
caseids_test = df['case_id'][df['test'] == 1].tolist()
caseids_test = [str(caseid).zfill(3) for caseid in caseids_test]
print(caseids_test)


for caseid in caseids_test:
    volume_array = load_slices(pred_dir, caseid)
    save_path = os.path.join(save_dir, caseid + '_fibrosis_pred.nii.gz')
    convert_to_nifti(volume_array, save_path)
    print(f"Saved {caseid} to {save_path}")

['033', '064', '065', '066', '067', '127', '128', '129', '192', '193']
Saved 033 to ../data/OSIC/processed/3d_visual/033_fibrosis_pred.nii.gz
Saved 064 to ../data/OSIC/processed/3d_visual/064_fibrosis_pred.nii.gz
Saved 065 to ../data/OSIC/processed/3d_visual/065_fibrosis_pred.nii.gz
Saved 066 to ../data/OSIC/processed/3d_visual/066_fibrosis_pred.nii.gz
Saved 067 to ../data/OSIC/processed/3d_visual/067_fibrosis_pred.nii.gz
Saved 127 to ../data/OSIC/processed/3d_visual/127_fibrosis_pred.nii.gz
Saved 128 to ../data/OSIC/processed/3d_visual/128_fibrosis_pred.nii.gz
Saved 129 to ../data/OSIC/processed/3d_visual/129_fibrosis_pred.nii.gz
Saved 192 to ../data/OSIC/processed/3d_visual/192_fibrosis_pred.nii.gz
Saved 193 to ../data/OSIC/processed/3d_visual/193_fibrosis_pred.nii.gz


## fibrosis_pred 3d visualization for wsss_coin

In [ ]:
# fibrosis_pred 3d visualization for wsss_coin
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd
import sys
sys.path.append('../')
from utils.save_nii import load_slices, convert_to_nifti

# Folder with 2D slices
pred_dir = '../experiments/wsss_coin/results/OSIC/pred_mask'
save_dir = '../experiments/wsss_coin/results/OSIC/3d_visual'
os.makedirs(save_dir, exist_ok=True)

# caseid = '035'
df = pd.read_csv('../data/OSIC/doctor_category.csv')
caseids_test = df['case_id'][df['test'] == 1].tolist()
caseids_test = [str(caseid).zfill(3) for caseid in caseids_test]
print(caseids_test)


for caseid in caseids_test:
    volume_array = load_slices(pred_dir, caseid)
    save_path = os.path.join(save_dir, caseid + '_fibrosis_pred.nii.gz')
    convert_to_nifti(volume_array, save_path)
    print(f"Saved {caseid} to {save_path}")

['033', '064', '065', '066', '067', '127', '128', '129', '192', '193']
Saved 033 to ../experiments/wsss_coin/results/OSIC/3d_visual/033_fibrosis_pred.nii.gz
Saved 064 to ../experiments/wsss_coin/results/OSIC/3d_visual/064_fibrosis_pred.nii.gz
Saved 065 to ../experiments/wsss_coin/results/OSIC/3d_visual/065_fibrosis_pred.nii.gz
Saved 066 to ../experiments/wsss_coin/results/OSIC/3d_visual/066_fibrosis_pred.nii.gz
Saved 067 to ../experiments/wsss_coin/results/OSIC/3d_visual/067_fibrosis_pred.nii.gz
Saved 127 to ../experiments/wsss_coin/results/OSIC/3d_visual/127_fibrosis_pred.nii.gz
Saved 128 to ../experiments/wsss_coin/results/OSIC/3d_visual/128_fibrosis_pred.nii.gz
Saved 129 to ../experiments/wsss_coin/results/OSIC/3d_visual/129_fibrosis_pred.nii.gz
Saved 192 to ../experiments/wsss_coin/results/OSIC/3d_visual/192_fibrosis_pred.nii.gz
Saved 193 to ../experiments/wsss_coin/results/OSIC/3d_visual/193_fibrosis_pred.nii.gz


## YYF_30Case 3d visualization for wsss_unet

In [3]:
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd
import sys
sys.path.append('../')
from utils.save_nii import save_as_nii

# Folder with 2D slices
pred_dir = '../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256_cropped/pred_mask'
pred_dir = '../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/pred_mask'
save_dir = pred_dir.replace('pred_mask', '3d_visual')
os.makedirs(save_dir, exist_ok=True)

caseids_test = os.listdir(pred_dir)
    
for caseid in caseids_test:
    save_path = os.path.join(save_dir, caseid + '_pred.nii.gz')
    save_as_nii(caseid, save_path, pred_dir, subdir = True)

Saved 1_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/1_0000_pred.nii.gz
Saved 2_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/2_0000_pred.nii.gz
Saved 3_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/3_0000_pred.nii.gz
Saved 5_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/5_0000_pred.nii.gz
Saved AIIB23_145_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/AIIB23_145_0000_pred.nii.gz
Saved AIIB23_146_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/AIIB23_146_0000_pred.nii.gz
Saved AIIB23_147_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/AIIB23_147_0000_pred.nii.gz
Saved AIIB23_159_0000 to ../experiments/wsss_unet/results/YYF_30Case/preprocessed_size256/3d_visual/AIIB23_159_0000_pred.nii.gz
Saved AIIB23_18_0000 to ../experiments/wsss_unet

In [1]:
# fibrosis_pred 3d visualization for full_supervised_unet
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd
import sys
sys.path.append('../')
from utils.save_nii import load_slices, convert_to_nifti

# Folder with 2D slices
pred_dir = '../experiments/full_supervised_unet/results/OSIC/pred_mask'
save_dir = pred_dir.replace('pred_mask', '3d_visual')
os.makedirs(save_dir, exist_ok=True)

# caseid = '035'
df = pd.read_csv('../data/OSIC/doctor_category.csv')
caseids_test = df['case_id'][df['test'] == 1].tolist()
caseids_test = [str(caseid).zfill(3) for caseid in caseids_test]


for caseid in caseids_test:
    volume_array = load_slices(pred_dir, caseid)
    save_path = os.path.join(save_dir, caseid + '_fibrosis_pred.nii.gz')
    convert_to_nifti(volume_array, save_path)
    print(f"Saved {caseid} to {save_path}")

Saved 033 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/033_fibrosis_pred.nii.gz
Saved 064 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/064_fibrosis_pred.nii.gz
Saved 065 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/065_fibrosis_pred.nii.gz
Saved 066 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/066_fibrosis_pred.nii.gz
Saved 067 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/067_fibrosis_pred.nii.gz
Saved 127 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/127_fibrosis_pred.nii.gz
Saved 128 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/128_fibrosis_pred.nii.gz
Saved 129 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/129_fibrosis_pred.nii.gz
Saved 192 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/192_fibrosis_pred.nii.gz
Saved 193 to ../experiments/full_supervised_unet/results/OSIC/3d_visual/193_fibrosis_pred.nii.gz


### CT new datasets 3d visualization - saved masks are stored in subfolders

In [4]:
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image
import pandas as pd
import sys
sys.path.append('../')
from utils.save_nii import save_as_nii

# Folder with 2D slices
# pred_dir = '../data/CT/processed'
# pred_dir = '../experiments/full_supervised_unet/results/CT/pred_mask'
pred_dir = '../experiments/wsss_unet/results/CT/pred_mask'
save_dir = pred_dir.replace(os.path.basename(pred_dir), '3d_visual')
os.makedirs(save_dir, exist_ok=True)

caseids_test = os.listdir(pred_dir)
    
for caseid in caseids_test:
    if 'pred_mask' in os.path.basename(pred_dir):
        save_path = os.path.join(save_dir, caseid + '_pred.nii.gz')
    else:
        save_path = os.path.join(save_dir, caseid + '_processed.nii.gz')
    save_as_nii(caseid, save_path, pred_dir, subdir = True)

Saved 0513585 to ../experiments/wsss_unet/results/CT/3d_visual/0513585_pred.nii.gz
Saved 0513666 to ../experiments/wsss_unet/results/CT/3d_visual/0513666_pred.nii.gz
Saved 0514090 to ../experiments/wsss_unet/results/CT/3d_visual/0514090_pred.nii.gz
Saved 0516435 to ../experiments/wsss_unet/results/CT/3d_visual/0516435_pred.nii.gz
Saved 0518580 to ../experiments/wsss_unet/results/CT/3d_visual/0518580_pred.nii.gz
Saved 0520477 to ../experiments/wsss_unet/results/CT/3d_visual/0520477_pred.nii.gz
Saved 0524287 to ../experiments/wsss_unet/results/CT/3d_visual/0524287_pred.nii.gz
Saved 0526150 to ../experiments/wsss_unet/results/CT/3d_visual/0526150_pred.nii.gz
Saved 0527090 to ../experiments/wsss_unet/results/CT/3d_visual/0527090_pred.nii.gz
Saved 0529802 to ../experiments/wsss_unet/results/CT/3d_visual/0529802_pred.nii.gz
Saved 0532860 to ../experiments/wsss_unet/results/CT/3d_visual/0532860_pred.nii.gz
Saved 0600860 to ../experiments/wsss_unet/results/CT/3d_visual/0600860_pred.nii.gz
Save